# Agenda HKBP: Segmentasi Kalimat

Notebook ini memecah enam bagian Agenda HKBP (sisi Batak Toba dan sisi Indonesia) menjadi **unit kalimat yang diberi keterangan struktur**, sebagai bahan penyelarasan sub-korpus paralel religius.

Penelitian: *Digitalisasi dan Praproses Korpus Bahasa Batak Toba sebagai Fondasi Pengembangan Model Bahasa Lokal Berbasis Deep Learning* (Skema PDP, Universitas Pelita Harapan).

Notebook ini **hanya** mengerjakan segmentasi. Penyelarasan, deduplikasi, pembagian train/dev/test, dan pemeriksaan sampel oleh manusia akan dikerjakan di tahap atau notebook berikutnya.

## Daftar isi

| Bagian | Isi | Tahap |
|---|---|---|
| 0 | Peta alur dan riwayat keputusan | |
| A | Persiapan, daftar input, konfigurasi, fungsi bantu | 0 |
| B | Segmentasi: validasi, struktur baris, kalimat, normalisasi OCR, pemeriksaan | a01 sampai a05 |
| C | Ringkasan, keterbatasan, dan langkah berikutnya | |

## Prinsip

1. **Tidak ada teks yang hilang.** Setiap unit menyimpan teks aslinya. Tahap a05 membuktikan bahwa gabungan semua unit sama persis dengan teks dokumen (selain spasi).
2. **Koreksi OCR terbatas dan tercatat.** Teks hasil koreksi disimpan di kolom terpisah. Setiap aturan yang mengubah unit tercatat namanya.
3. **Parameter ditetapkan sebelum dijalankan.** Semua parameter ada di sel Konfigurasi.
4. **Titik simpan (checkpoint).** Tahap yang masukan, parameter, dan kodenya tidak berubah diambil dari checkpoint saat dijalankan ulang.
5. **Tidak butuh GPU.** Seluruh notebook selesai dalam kurang dari satu menit di runtime CPU biasa.

## Catatan hak cipta

Agenda HKBP adalah terbitan resmi HKBP, dan izin redistribusinya belum dipastikan. Keluaran notebook ini memuat teks Agenda, jadi **simpan hanya di Google Drive tim**. Jangan unggah keluarannya ke repositori publik sampai izinnya jelas.

---
# Bagian 0. Peta alur dan riwayat keputusan

## Peta alur

Jalankan sel di bawah ini untuk menampilkan diagram. Sel ini tidak membutuhkan pustaka tambahan maupun akses ke Google Drive.

- **Kotak biru** adalah tahap di notebook ini.
- **Kotak hijau** adalah keluaran yang dipakai tahap berikutnya.
- **Kotak abu-abu bergaris putus** adalah pekerjaan yang sengaja belum dikerjakan di sini.

In [ ]:
from IPython.display import SVG, display

PETA_ALUR = r"""<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 760 900' width='100%' style='max-width:760px' role='img' aria-label='Peta alur segmentasi Agenda HKBP'><defs><marker id='ujung' viewBox='0 0 10 10' refX='9' refY='5' markerWidth='7' markerHeight='7' orient='auto-start-reverse'><path d='M0,0 L10,5 L0,10 z' fill='#6B7280'/></marker></defs><rect x='0' y='0' width='760' height='900' fill='#FFFFFF'/><text x='380.0' y='34' font-family='Helvetica, Arial, sans-serif' font-size='19' font-weight='700' fill='#111827' text-anchor='middle'>Peta alur segmentasi Agenda HKBP</text><rect x='180' y='60' width='400' height='64' rx='10' fill='#F9FAFB' stroke='#6B7280' stroke-width='1.6'/><text x='380.0' y='87.5' font-family='Helvetica, Arial, sans-serif' font-size='13.5' font-weight='700' fill='#111827' text-anchor='middle'>Masukan: btb_multigenre.csv</text><text x='380.0' y='104.5' font-family='Helvetica, Arial, sans-serif' font-size='11.5' font-weight='400' fill='#374151' text-anchor='middle'>rilis v2, commit 35068ec, sidik SHA-256 diperiksa</text><line x1='380.0' y1='124' x2='380.0' y2='156' stroke='#6B7280' stroke-width='1.6' marker-end='url(#ujung)'/><rect x='180' y='160' width='400' height='64' rx='10' fill='#EEF3FC' stroke='#3B6FD8' stroke-width='1.6'/><text x='380.0' y='187.5' font-family='Helvetica, Arial, sans-serif' font-size='13.5' font-weight='700' fill='#111827' text-anchor='middle'>a01. Muat dan validasi Agenda HKBP</text><text x='380.0' y='204.5' font-family='Helvetica, Arial, sans-serif' font-size='11.5' font-weight='400' fill='#374151' text-anchor='middle'>6 dokumen (mg_054 sampai mg_059), 35.097 kata Toba</text><line x1='380.0' y1='224' x2='380.0' y2='256' stroke='#6B7280' stroke-width='1.6' marker-end='url(#ujung)'/><rect x='180' y='260' width='400' height='64' rx='10' fill='#EEF3FC' stroke='#3B6FD8' stroke-width='1.6'/><text x='380.0' y='287.5' font-family='Helvetica, Arial, sans-serif' font-size='13.5' font-weight='700' fill='#111827' text-anchor='middle'>a02. Struktur baris</text><text x='380.0' y='304.5' font-family='Helvetica, Arial, sans-serif' font-size='11.5' font-weight='400' fill='#374151' text-anchor='middle'>judul bagian, subjudul, label peran, penanda butir, gabung baris</text><line x1='380.0' y1='324' x2='380.0' y2='356' stroke='#6B7280' stroke-width='1.6' marker-end='url(#ujung)'/><rect x='180' y='360' width='400' height='64' rx='10' fill='#EEF3FC' stroke='#3B6FD8' stroke-width='1.6'/><text x='380.0' y='387.5' font-family='Helvetica, Arial, sans-serif' font-size='13.5' font-weight='700' fill='#111827' text-anchor='middle'>a03. Segmentasi kalimat</text><text x='380.0' y='404.5' font-family='Helvetica, Arial, sans-serif' font-size='11.5' font-weight='400' fill='#374151' text-anchor='middle'>kurung dilindungi, seruan dan vokatif digabung</text><line x1='380.0' y1='424' x2='380.0' y2='456' stroke='#6B7280' stroke-width='1.6' marker-end='url(#ujung)'/><rect x='180' y='460' width='400' height='64' rx='10' fill='#EEF3FC' stroke='#3B6FD8' stroke-width='1.6'/><text x='380.0' y='487.5' font-family='Helvetica, Arial, sans-serif' font-size='13.5' font-weight='700' fill='#111827' text-anchor='middle'>a04. Normalisasi OCR terbatas dan jenis unit</text><text x='380.0' y='504.5' font-family='Helvetica, Arial, sans-serif' font-size='11.5' font-weight='400' fill='#374151' text-anchor='middle'>teks asli tetap disimpan, setiap koreksi tercatat</text><line x1='380.0' y1='524' x2='380.0' y2='556' stroke='#6B7280' stroke-width='1.6' marker-end='url(#ujung)'/><rect x='180' y='560' width='400' height='64' rx='10' fill='#EEF3FC' stroke='#3B6FD8' stroke-width='1.6'/><text x='380.0' y='587.5' font-family='Helvetica, Arial, sans-serif' font-size='13.5' font-weight='700' fill='#111827' text-anchor='middle'>a05. Pemeriksaan dan ringkasan</text><text x='380.0' y='604.5' font-family='Helvetica, Arial, sans-serif' font-size='11.5' font-weight='400' fill='#374151' text-anchor='middle'>keutuhan teks, kesejajaran bagian, pembanding s04, sampel</text><line x1='380.0' y1='624' x2='380.0' y2='656' stroke='#2F855A' stroke-width='1.6' marker-end='url(#ujung)'/><rect x='180' y='660' width='400' height='80' rx='10' fill='#EDF7F0' stroke='#2F855A' stroke-width='1.6'/><text x='380.0' y='687.0' font-family='Helvetica, Arial, sans-serif' font-size='13.5' font-weight='700' fill='#111827' text-anchor='middle'>Keluaran utama</text><text x='380.0' y='704.0' font-family='Helvetica, Arial, sans-serif' font-size='11.5' font-weight='400' fill='#374151' text-anchor='middle'>a04_unit_agenda.csv: satu baris per unit</text><text x='380.0' y='721.0' font-family='Helvetica, Arial, sans-serif' font-size='11.5' font-weight='400' fill='#374151' text-anchor='middle'>a05_per_bagian.csv, a05_sampel_periksa.csv, a05_ringkasan.json</text><line x1='380.0' y1='740' x2='380.0' y2='776' stroke='#6B7280' stroke-width='1.6' stroke-dasharray='5 4' marker-end='url(#ujung)'/><rect x='120' y='780' width='520' height='80' rx='10' fill='#F3F4F6' stroke='#9CA3AF' stroke-width='1.6' stroke-dasharray='6 4'/><text x='380.0' y='807.0' font-family='Helvetica, Arial, sans-serif' font-size='13.5' font-weight='700' fill='#111827' text-anchor='middle'>Belum dikerjakan di notebook ini</text><text x='380.0' y='824.0' font-family='Helvetica, Arial, sans-serif' font-size='11.5' font-weight='400' fill='#374151' text-anchor='middle'>penyelarasan berjangkar per bagian, deduplikasi, pembagian train/dev/test,</text><text x='380.0' y='841.0' font-family='Helvetica, Arial, sans-serif' font-size='11.5' font-weight='400' fill='#374151' text-anchor='middle'>dan pemeriksaan sampel oleh manusia</text></svg>"""

display(SVG(data=PETA_ALUR))

## Riwayat keputusan

| No | Peristiwa | Keputusan dan alasannya |
|---|---|---|
| 1 | Agenda HKBP (enam bagian, 35.097 kata Toba) masuk `btb_multigenre.csv` sebagai teks dokumen utuh | Agenda **dikeluarkan dari korpus paralel utama**, karena porsinya terlalu besar (59% kata Toba multi-genre) dan strukturnya tidak sejajar baris per baris |
| 2 | Uji penyelarasan LaBSE per bagian di pipeline v2 (tanpa jangkar) | Menghasilkan 2.601 pasangan, 558 (21%) berprioritas tinggi, dan 566 duplikat persis. Banyak judul rubrik yang terpasang dengan kalimat. Hasil ini tidak dipakai |
| 3 | Agenda dan Alkitab akan dirilis sebagai **sub-korpus paralel religius terpisah** | Supaya korpus utama tidak didominasi teks religius dan porsi data tervalidasi tidak turun |
| 4 | Pemeriksaan ulang struktur Agenda | Butir bernomor sisi Toba dan Indonesia 96,7% sejajar urutannya (353 dari 365 butir Toba), dan jumlah judul bagian berangka Romawi sama di setiap dokumen. Jadi Agenda lebih tepat disebut **terjemahan dengan rubrik yang tidak sejajar**, bukan dua teks yang tidak saling menerjemahkan. Struktur ini akan dipakai sebagai jangkar penyelarasan |
| 5 | Notebook ini | Segmentasi dikerjakan lebih dulu dan terpisah. Setiap unit diberi keterangan bagian, subbagian, butir, label peran, dan jenis unit, supaya penyelarasan nanti bisa dibatasi per bagian |
| 6 | Sisi Indonesia hasil OCR dari terbitan cetak | Koreksi OCR hanya untuk pola yang jelas dan daftar kata eksplisit. Ejaan Toba tidak dikoreksi, karena tidak ada pembanding yang dapat dipercaya |

---
# Bagian A. Persiapan

## Tahap 0. Persiapan lingkungan

**Tujuan:** menyiapkan lingkungan kerja yang sama setiap kali notebook dijalankan.

**Yang dikerjakan:**
1. Menghubungkan Google Drive bila notebook berjalan di Colab.
2. Membaca konfigurasi, menyiapkan folder kerja, dan memastikan berkas masukan tersedia.
3. Mendefinisikan fungsi bantu checkpoint dan fungsi bantu teks.

Tidak ada pustaka tambahan yang perlu dipasang. `pandas` dan `numpy` sudah tersedia di Colab.

## Daftar input

Hanya satu berkas yang dibutuhkan:

```
btb-agenda-hkbp/
└── input/
    └── btb_multigenre.csv
```

| File | Asal | Dipakai di |
|---|---|---|
| `btb_multigenre.csv` | Repositori GitHub, `data/btb_multigenre.csv` (korpus multi-genre v2) | Tahap a01 |

Bila berkas belum ada di folder `input/`, notebook mengunduhnya dari GitHub pada commit yang dikunci di Konfigurasi (`url_multigenre`), lalu memeriksa sidik SHA-256-nya. Dengan begitu, notebook selalu memakai versi data yang sama.

In [ ]:
import os
import re
import json
import random
import hashlib
import platform
import unicodedata
import urllib.request
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from google.colab import drive
    DI_COLAB = True
except ImportError:
    DI_COLAB = False

if DI_COLAB:
    drive.mount("/content/drive")

pd.set_option("display.max_colwidth", 100)
print("Berjalan di Colab:", DI_COLAB)
print("Python", platform.python_version(), "| pandas", pd.__version__)

## Konfigurasi

Seluruh parameter ada di satu sel ini. Setiap tahap mencatat sidik dari parameter, berkas masukan, dan kode yang dipakainya. Bila salah satunya berubah, tahap itu dan tahap sesudahnya otomatis dihitung ulang saat **Run all**.

| Bagian | Isi | Dipakai di tahap |
|---|---|---|
| `folder_kerja` | Lokasi folder kerja di Google Drive | Semua |
| `berkas`, `url_multigenre`, `sha256_multigenre` | Nama berkas masukan, sumber unduhan, dan sidik yang diharapkan | a01 |
| `paksa_ulang` | Kode tahap yang dipaksa dihitung ulang, misalnya `["a03"]` | Semua |
| `dokumen_agenda` | Sub-genre dan `doc_id` Agenda, serta jumlah kata Toba yang diharapkan | a01 |
| `struktur` | Daftar label peran (dan label yang perannya tidak diteruskan), aturan judul bagian, subjudul, deret butir, dan penggabungan baris | a02 |
| `kalimat` | Daftar singkatan, perlakuan kurung panjang, dan penggabungan seruan | a03 |
| `ocr` | Aturan koreksi OCR dan daftar kata yang dikoreksi | a02 (pengenalan label), a04 |
| `jenis_unit` | Kata formula liturgi dan pola singkatan lanjutan (`d.u.`, `dst.`) | a04 |
| `pemeriksaan` | Ukuran sampel pemeriksaan dan pembanding segmentasi tahap s04 notebook utama | a05 |

**Cara mengubah aturan:** misalnya untuk mematikan penggabungan seruan, ubah `"aktif": True` menjadi `False` di `kalimat` > `gabung_seruan`, lalu pilih **Run all**. Tahap a03 sampai a05 akan dihitung ulang.

In [ ]:
KONFIGURASI = {
    "versi_pipeline": "agenda-seg-1.0.0",
    "folder_kerja": os.environ.get("BTB_FOLDER_KERJA", "/content/drive/MyDrive/Datasets/btb-agenda-hkbp"),
    "seed": 42,
    "paksa_ulang": [],

    "berkas": {"multigenre": "btb_multigenre.csv"},
    "unduh_bila_tidak_ada": True,
    "url_multigenre": ("https://raw.githubusercontent.com/FeliksMakarios/batak-toba-korpus/"
                       "35068eca411be09e2a90ca05110b6b00487f3c04/data/btb_multigenre.csv"),
    # Isi None bila sengaja memakai versi btb_multigenre.csv yang lain.
    "sha256_multigenre": "c316c145c921031c886f8f35ccc6d9ea9323192b9ba37f2932a5733184fa7f59",

    "dokumen_agenda": {
        "subgenre": "Ritual Text",
        "doc_id": ["mg_054", "mg_055", "mg_056", "mg_057", "mg_058", "mg_059"],
        "kata_bt_diharapkan": 35097,
    },

    "struktur": {
        # Label peran (rubrik) dikenali bila muncul di awal baris dan diikuti titik dua.
        # Pencocokan memakai huruf kecil, setelah huruf berspasi digabung (misalnya "D O A" menjadi "doa"),
        # dan setelah angka Romawi di belakang label dibuang (misalnya "Paragenda II").
        "label_peran": {
            "bbc": [
                "marende", "marende huria", "marende huria huhut papungu pelean", "marende sikola",
                "marende nasida", "marende mardongan giringgiring", "paragenda", "huria", "huria mandok",
                "martangiang", "tangiang", "pasupasu", "pasu-pasu", "pasunasu", "alusna", "alus",
                "marjamita", "jamita", "pandita", "pandita mandok", "holan pandita", "ephorus mandok",
                "pandidion", "hata sipaingot", "tingting", "endehononhon", "haporseaon", "haporseson",
                "ayat sidohonon",
            ],
            "ind": [
                "nyanyian bersama", "nyanyian persembahan", "pelayan liturgi", "doa", "doa bapa kami",
                "berkat", "berkat baptisan", "berkat perkawinan", "jawab", "jawabnya", "jemaat menyambut",
                "pendeta", "oleh pendeta", "ephorus", "pembaptisan", "nasihat dan bimbingan",
                "nasihat dan bimbingan calon baptisan", "bimbingan perjamuan", "warta jemaat", "khotbah",
                "ayat bacaan", "paduan suara", "pengakuan iman", "doa pengucapan syukur kepada allah",
            ],
        },
        # Label yang hanya menandai tempat nyanyian, warta, atau khotbah (isinya tidak dicetak).
        # Perannya tidak diteruskan ke baris-baris sesudahnya.
        "label_tidak_menurun": {
            "bbc": ["marende", "marende huria", "marende huria huhut papungu pelean", "marende sikola",
                    "marende nasida", "marende mardongan giringgiring", "tingting", "jamita", "marjamita"],
            "ind": ["nyanyian bersama", "nyanyian persembahan", "warta jemaat", "khotbah", "paduan suara"],
        },
        "judul_besar_awalan": ["bagian"],
        "judul_besar_maks_kata": 4,
        "judul_bagian_maks_kata": 12,
        "subjudul_maks_kata": 10,
        # Baris berpenanda Romawi yang berurutan nilainya (I, II, III, ...) di baris-baris yang bersebelahan
        # dianggap butir daftar (misalnya Sepuluh Hukum), bukan judul bagian.
        "deret_daftar_min": 3,
        "gabung_baris": {"akhir_koma": True, "lanjut_huruf_kecil": True},
    },

    "kalimat": {
        "singkatan": [
            "no", "hal", "dst", "dll", "dsb", "bnd", "ay", "pdt", "st", "sdr", "bp", "ibu",
            "jes", "yes", "mat", "mrk", "mark", "luk", "joh", "yoh", "kis", "rom", "kor", "gal",
            "ef", "fil", "kol", "tes", "tim", "tit", "ibr", "yak", "ptr", "why", "kej", "kel",
            "im", "bil", "ul", "mzm", "ps", "ams", "pkh", "yer", "jer", "yeh", "hos", "mik", "sef",
        ],
        # Isi kurung yang lebih panjang dari ini (misalnya doa pilihan dalam kurung) tetap dipecah per kalimat.
        "kurung_panjang_min_kata": 25,
        "gabung_seruan": {
            "aktif": True,
            "maks_kata_pendek": 3,
            "awalan_vokatif": {"bbc": ["ale", "o", "oi"], "ind": ["ya", "hai", "wahai", "o"]},
            "maks_kata_vokatif": 12,
        },
    },

    "ocr": {
        "hapus_tanda_negasi": True,
        "gabung_huruf_berspasi": True,
        "huruf_l_dalam_kapital": True,
        "ringkas_titik_isian": True,
        "la_menjadi_Ia": True,
        "ganti_literal": {"bbc": {"(loar": "Goar"}, "ind": {"A mi n": "Amin", "A m in": "Amin"}},
        "ganti_kata": {
            "bbc": {},
            "ind": {
                "Jemeat": "Jemaat", "jemeat": "jemaat", "penderitean": "penderitaan",
                "penderitann": "penderitaan", "Kerajean": "Kerajaan", "kerajean": "kerajaan",
                "Kerajaun": "Kerajaan", "pertanyean": "pertanyaan", "kebijaksanean": "kebijaksanaan",
                "kependetean": "kependetaan", "kependetann": "kependetaan", "perkataun": "perkataan",
                "Kepercayaun": "Kepercayaan", "kepercayean": "kepercayaan", "Tuhnn": "Tuhan",
                "demikinn": "demikian", "pencobaun": "pencobaan", "pencobean": "pencobaan",
                "pekerjean": "pekerjaan", "selurnh": "seluruh", "kesediaun": "kesediaan",
                "menyurnh": "menyuruh", "pengadean": "pengadaan", "kesengsarean": "kesengsaraan",
                "keadean": "keadaan", "kelegean": "kelegaan", "Perjamnan": "Perjamuan",
                "hokumMu": "hukumMu", "persembanan": "persembahan",
            },
        },
    },

    "jenis_unit": {
        "formula": ["amen", "amin", "haleluya", "halleluya"],
        "penanda_singkatan_lanjutan": [r"\bd\.\s?u\.", r"\bdst\b", r"\bd\.\s?s\.\s?t\."],
    },

    "pemeriksaan": {
        "sampel_per_jenis": 15,
        "unit_terpanjang": 10,
        # Daftar singkatan tahap s04 notebook utama, untuk membandingkan jumlah kalimat dengan segmentasi lama.
        "singkatan_s04": [
            "dr", "drs", "dra", "ir", "prof", "hj", "h", "tn", "ny", "sdr",
            "yth", "dll", "dst", "dsb", "tsb", "tgl", "no", "hlm",
            "jl", "jln", "kel", "kec", "kab", "prov",
            "st", "sd", "smp", "sma", "smk", "pt", "cv", "tbk",
            "op", "ompu", "amang", "inang", "ama", "ina",
            "vs", "etc", "vol",
        ],
    },
}

random.seed(KONFIGURASI["seed"])
np.random.seed(KONFIGURASI["seed"])

ROOT = Path(KONFIGURASI["folder_kerja"])
DIR_INPUT = ROOT / "input"
DIR_OUT = ROOT / "keluaran"
DIR_META = DIR_OUT / "_meta"
for d in (DIR_INPUT, DIR_OUT, DIR_META):
    d.mkdir(parents=True, exist_ok=True)

BERKAS_MULTIGENRE = DIR_INPUT / KONFIGURASI["berkas"]["multigenre"]
if not BERKAS_MULTIGENRE.exists():
    if not KONFIGURASI["unduh_bila_tidak_ada"]:
        raise FileNotFoundError(f"Letakkan '{BERKAS_MULTIGENRE.name}' di folder {DIR_INPUT} lalu jalankan ulang.")
    print("Berkas belum ada di folder input. Mengunduh dari GitHub ...")
    urllib.request.urlretrieve(KONFIGURASI["url_multigenre"], BERKAS_MULTIGENRE)

print("Folder kerja:", ROOT)
print("Berkas masukan:", BERKAS_MULTIGENRE, "| ada:", BERKAS_MULTIGENRE.exists())

## Fungsi bantu checkpoint

Setiap tahap dibungkus objek `Tahap`, sama dengan notebook utama. Objek ini menghitung **sidik** dari parameter tahap, isi berkas masukan (SHA-256), kode di sel tahap itu, dan versi pipeline. Sidik disimpan di `keluaran/_meta/aNN.json` bersama daftar keluaran dan ringkasannya.

Saat **Run all**, sebuah tahap dilewati (diambil dari checkpoint) hanya bila:
1. semua berkas keluarannya masih ada,
2. sidiknya sama dengan sidik yang tersimpan, dan
3. kode tahap tidak tercantum di `paksa_ulang`.

Berbeda dengan notebook utama, semua tabel di sini dibaca sebagai teks (`dtype=str`) supaya isian kosong tetap berupa string kosong, bukan `NaN`.

In [ ]:
STATUS_TAHAP = {}


def sha256_berkas(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for blok in iter(lambda: f.read(1 << 20), b""):
            h.update(blok)
    return h.hexdigest()


def sidik_objek(obj):
    teks = json.dumps(obj, sort_keys=True, ensure_ascii=False, default=str)
    return hashlib.sha256(teks.encode("utf-8")).hexdigest()


def sumber_sel_aktif():
    try:
        return get_ipython().user_ns["_ih"][-1]
    except Exception:
        return ""


def baca_csv(path, kolom_angka=()):
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    for kol in kolom_angka:
        if kol in df.columns:
            df[kol] = pd.to_numeric(df[kol]).astype(int)
    return df


def simpan_csv(df, path):
    df.to_csv(path, index=False, encoding="utf-8")


def _ke_json(o):
    return o.item() if hasattr(o, "item") else str(o)


def simpan_json(obj, path):
    Path(path).write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=_ke_json), encoding="utf-8")


class Tahap:
    def __init__(self, kode, nama, masukan, keluaran, parameter):
        self.kode = kode
        self.nama = nama
        self.masukan = [Path(p) for p in masukan]
        self.keluaran = [Path(p) for p in keluaran]
        self.parameter = parameter
        hilang = [str(p) for p in self.masukan if not p.exists()]
        if hilang:
            raise FileNotFoundError(f"[{kode}] Berkas masukan tidak ditemukan: {hilang}")
        self.sidik_masukan = {p.name: sha256_berkas(p) for p in self.masukan}
        self.sidik_kode = sidik_objek(sumber_sel_aktif())
        self.sidik = sidik_objek({
            "versi": KONFIGURASI["versi_pipeline"],
            "parameter": parameter,
            "masukan": self.sidik_masukan,
            "kode_sel": self.sidik_kode,
        })
        self.path_meta = DIR_META / f"{kode}.json"

    def bisa_dilewati(self):
        if self.kode in KONFIGURASI["paksa_ulang"]:
            return False
        if not self.path_meta.exists() or not all(p.exists() for p in self.keluaran):
            return False
        meta = json.loads(self.path_meta.read_text(encoding="utf-8"))
        if meta.get("sidik") != self.sidik:
            return False
        STATUS_TAHAP[self.kode] = "checkpoint"
        print(f"[{self.kode}] {self.nama}: diambil dari checkpoint ({meta['waktu']}).")
        for k, v in meta.get("ringkasan", {}).items():
            print(f"    {k}: {v}")
        return True

    def selesai(self, ringkasan):
        meta = {
            "kode": self.kode,
            "nama": self.nama,
            "sidik": self.sidik,
            "waktu": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "versi_pipeline": KONFIGURASI["versi_pipeline"],
            "parameter": self.parameter,
            "masukan": self.sidik_masukan,
            "keluaran": {p.name: sha256_berkas(p) for p in self.keluaran},
            "ringkasan": ringkasan,
        }
        simpan_json(meta, self.path_meta)
        STATUS_TAHAP[self.kode] = "dijalankan"
        print(f"[{self.kode}] {self.nama}: selesai dan disimpan.")
        for k, v in ringkasan.items():
            print(f"    {k}: {v}")


def tampilkan(df, n=5):
    try:
        display(df.head(n))
    except NameError:
        print(df.head(n).to_string())

## Fungsi bantu teks

Fungsi di sel ini dipakai oleh beberapa tahap: penghitung kata, normalisasi OCR, dan pembuat kunci label. Sidik kode sel ini (`SIDIK_FUNGSI_TEKS`) ikut dimasukkan ke parameter tahap a02 sampai a05. Jadi bila fungsi di sini diubah, tahap-tahap itu ikut dihitung ulang.

**Aturan normalisasi OCR** (semuanya bisa dimatikan di Konfigurasi bagian `ocr`):

| Aturan | Contoh | Alasan |
|---|---|---|
| `literal` | `(loar` menjadi `Goar`, `A mi n` menjadi `Amin` | Salah baca OCR yang tidak tertangkap aturan umum |
| `tanda_negasi` | `uhum¬Na` menjadi `uhumNa`, `anggota¬ anggota` menjadi `anggota-anggota` | Sisa tanda pemenggalan kata dari berkas cetak |
| `i_tanpa_titik` | `Tangıang` menjadi `Tangiang` | Karakter Unicode yang tampak sama tetapi berbeda |
| `huruf_berspasi` | `D O A` menjadi `DOA`, `A m i n .` menjadi `Amin .`, `P E N D ETA` menjadi `PENDETA` | Huruf renggang pada judul dan rubrik. Hanya untuk deret minimal tiga huruf tunggal |
| `l_dalam_kapital` | `NYANYlAN` menjadi `NYANYIAN`, `Dl` menjadi `DI` | Huruf l kecil terbaca di tengah kata berhuruf kapital |
| `titik_isian` | `No..........` menjadi `No...` | Deret titik isian yang panjangnya berbeda-beda, supaya varian yang sama dapat dikenali |
| `kata` | `Kerajean` menjadi `Kerajaan`, `selurnh` menjadi `seluruh` | Daftar kata eksplisit hasil pemeriksaan sisi Indonesia. Tidak ada koreksi berbasis pola |
| `la_menjadi_Ia` | `Lalu la memeluk` menjadi `Lalu Ia memeluk` | Kata ganti `Ia` terbaca `la`. Hanya sisi Indonesia, hanya bila diikuti kata berhuruf kecil |

Ejaan sisi Toba **tidak** dikoreksi, kecuali `(loar` (huruf G terbaca sebagai kurung buka). Dugaan salah ketik Toba seperti `nueeng` atau `jula` dibiarkan, karena tidak ada pembanding yang dapat dipercaya.

In [ ]:
def jumlah_kata(s):
    return len(re.findall(r"\w+", s)) if isinstance(s, str) else 0


_TOKEN_HURUF = re.compile(r"[(\"“‘']?[A-Za-z0][)\"”’'.,:;!?]*")


def gabung_huruf_berspasi(teks):
    token = teks.split(" ")
    hasil, i, n = [], 0, len(token)
    while i < n:
        if not _TOKEN_HURUF.fullmatch(token[i]):
            hasil.append(token[i])
            i += 1
            continue
        j = i
        while (j < n and _TOKEN_HURUF.fullmatch(token[j])
               and (j == i or not re.search(r"[)\"”’'.,:;!?]$", token[j - 1]))
               and (j == i or not re.match(r"[(\"“‘']", token[j]))):
            j += 1
        deret = token[i:j]
        if len(deret) < 3:
            hasil.extend(deret)
            i = j
            continue
        awal = re.match(r"[(\"“‘']*", deret[0]).group(0)
        akhir = re.search(r"[)\"”’'.,:;!?]*$", deret[-1]).group(0)
        huruf = "".join(re.sub(r"[^A-Za-z0]", "", t) for t in deret)
        kapital = huruf.replace("0", "O").isupper()
        if not kapital and "0" in huruf:
            hasil.extend(deret)
            i = j
            continue
        if kapital:
            huruf = huruf.replace("0", "O")
            # Lekatkan potongan kapital pendek di sebelahnya, misalnya "P E N D ETA" atau "BE R K A T".
            if not akhir and j < n and re.fullmatch(r"[A-Z]{2,3}[.,:;!?]*", token[j]):
                akhir = re.search(r"[.,:;!?]*$", token[j]).group(0)
                huruf += token[j][: len(token[j]) - len(akhir)]
                j += 1
            if not awal and hasil and re.fullmatch(r"[A-Z]{2,3}", hasil[-1]):
                huruf = hasil.pop() + huruf
        hasil.append(awal + huruf + akhir)
        i = j
    return " ".join(hasil)


def normalkan_ocr(teks, bahasa, p=None):
    p = p or KONFIGURASI["ocr"]
    catatan = []
    t = teks

    def terapkan(nama, baru):
        nonlocal t
        if baru != t:
            catatan.append(nama)
            t = baru

    for lama, baru in p["ganti_literal"].get(bahasa, {}).items():
        terapkan(f"literal:{lama}", t.replace(lama, baru))
    if p["hapus_tanda_negasi"]:
        terapkan("tanda_negasi", re.sub(r"¬", "", re.sub(r"¬\s+", "-", t.replace("-¬", "-"))))
    terapkan("i_tanpa_titik", t.replace("ı", "i"))
    if p["gabung_huruf_berspasi"]:
        terapkan("huruf_berspasi", gabung_huruf_berspasi(t))
    if p["huruf_l_dalam_kapital"]:
        terapkan("l_dalam_kapital", re.sub(r"\b(?=[A-Zl]*[A-Z])(?=[A-Zl]*l)[A-Zl]{2,}\b",
                                           lambda m: m.group(0).replace("l", "I"), t))
    if p["ringkas_titik_isian"]:
        terapkan("titik_isian", re.sub(r"\.{4,}", "...", t))
    for lama, baru in p["ganti_kata"].get(bahasa, {}).items():
        terapkan(f"kata:{lama}", re.sub(rf"\b{re.escape(lama)}\b", baru, t))
    if p["la_menjadi_Ia"] and bahasa == "ind":
        terapkan("la_menjadi_Ia", re.sub(r"\bla\b(?= [a-z])", "Ia", t))
    return t, catatan


def kunci_label(teks):
    t, _ = normalkan_ocr(teks, "ind")
    t = re.sub(r"\s+", " ", t.lower()).strip()
    return re.sub(r"\s+(?:i|ii|iii|iv)$", "", t)


SIDIK_FUNGSI_TEKS = sidik_objek(sumber_sel_aktif())
print("Contoh normalisasi:")
for contoh, bhs in [("D O A :", "ind"), ("A m i n .", "ind"), ("P E N D ETA :", "ind"), ("NYANYlAN BERSAMA : No..........", "ind"),
                    ("hona uhum¬Na do", "bbc"), ("Lalu la memeluk anakanak itu", "ind"), ("Kerajean Allah", "ind")]:
    print(f"    {contoh!r:40} -> {normalkan_ocr(contoh, bhs)[0]!r}")

---
# Bagian B. Segmentasi Agenda HKBP

| Tahap | Isi | Keluaran utama |
|---|---|---|
| a01 | Muat dan validasi enam dokumen Agenda | `a01_agenda_dokumen.csv`, `a01_validasi.json` |
| a02 | Struktur baris: judul, subjudul, label peran, penanda butir, penggabungan baris | `a02_baris.csv`, `a02_ringkasan_struktur.csv` |
| a03 | Segmentasi kalimat di dalam setiap baris | `a03_unit_mentah.csv` |
| a04 | Normalisasi OCR terbatas dan jenis unit | `a04_unit_agenda.csv`, `a04_log_ocr.csv` |
| a05 | Pemeriksaan keutuhan, kesejajaran bagian, pembanding segmentasi lama, dan sampel | `a05_ringkasan.json`, `a05_per_bagian.csv`, `a05_sampel_periksa.csv` |

Kode bahasa mengikuti notebook utama: `bbc` untuk Batak Toba dan `ind` untuk Indonesia.

---
## Tahap a01. Muat dan validasi Agenda HKBP

**Tujuan:** memastikan notebook memakai versi data yang tepat, dan keenam bagian Agenda terbaca utuh di kedua bahasa.

**Yang dikerjakan:**
1. Memeriksa sidik SHA-256 `btb_multigenre.csv` terhadap nilai di Konfigurasi. Bila berbeda, tahap ini berhenti dengan pesan galat. Isi `sha256_multigenre` dengan `None` bila memang sengaja memakai versi lain.
2. Mengambil dokumen bersub-genre `Ritual Text`, lalu memeriksa bahwa `doc_id`-nya tepat `mg_054` sampai `mg_059`.
3. Memeriksa bahwa kedua sisi tidak kosong, dan tidak ada teks sepanjang tepat 32.767 karakter (batas isi satu sel Excel, tanda kemungkinan terpotong).
4. Menghitung baris tidak kosong dan jumlah kata per dokumen, lalu membandingkan total kata Toba dengan angka di README (35.097).

**Masukan:** `input/btb_multigenre.csv`.

**Keluaran yang diharapkan:**
- `a01_agenda_dokumen.csv`: enam baris (`doc_id`, `title`, `text_bt`, `text_id`, `genre`, `subgenre`, `source_link`).
- `a01_validasi.json`: hasil pemeriksaan, termasuk jumlah baris dan kata per dokumen.

**Dipakai untuk:** tahap a02, dan sebagai pembanding keutuhan teks di tahap a05.

**Cara memeriksa:** total kata Toba harus 35.097 dan `kata_bt_sesuai` bernilai `True`.

In [ ]:
P_A01_DOK = DIR_OUT / "a01_agenda_dokumen.csv"
P_A01_VAL = DIR_OUT / "a01_validasi.json"
PARAM_DOK = KONFIGURASI["dokumen_agenda"]

t1 = Tahap(
    "a01", "Muat dan validasi Agenda HKBP",
    masukan=[BERKAS_MULTIGENRE],
    keluaran=[P_A01_DOK, P_A01_VAL],
    parameter={"dokumen": PARAM_DOK, "sha256_diharapkan": KONFIGURASI["sha256_multigenre"]},
)

if not t1.bisa_dilewati():
    sidik = t1.sidik_masukan[BERKAS_MULTIGENRE.name]
    if KONFIGURASI["sha256_multigenre"] and sidik != KONFIGURASI["sha256_multigenre"]:
        raise ValueError(
            f"Sidik {BERKAS_MULTIGENRE.name} ({sidik}) berbeda dengan yang diharapkan. "
            "Gunakan berkas dari commit yang dikunci, atau isi sha256_multigenre dengan None bila sengaja memakai versi lain."
        )
    mg = baca_csv(BERKAS_MULTIGENRE)
    agenda = mg[mg["subgenre"] == PARAM_DOK["subgenre"]].sort_values("doc_id").reset_index(drop=True)
    ditemukan = agenda["doc_id"].tolist()
    if ditemukan != PARAM_DOK["doc_id"]:
        raise ValueError(f"doc_id Agenda tidak sesuai. Diharapkan {PARAM_DOK['doc_id']}, ditemukan {ditemukan}.")

    per_dok = []
    for _, r in agenda.iterrows():
        per_dok.append({
            "doc_id": r["doc_id"], "title": r["title"],
            "baris_bt": sum(1 for b in r["text_bt"].split("\n") if b.strip()),
            "baris_id": sum(1 for b in r["text_id"].split("\n") if b.strip()),
            "kata_bt": jumlah_kata(r["text_bt"]), "kata_id": jumlah_kata(r["text_id"]),
        })
    kosong = [f"{r['doc_id']}:{k}" for _, r in agenda.iterrows() for k in ("text_bt", "text_id") if not r[k].strip()]
    terpotong = [f"{r['doc_id']}:{k}" for _, r in agenda.iterrows() for k in ("text_bt", "text_id") if len(r[k]) == 32767]
    kata_bt = sum(d["kata_bt"] for d in per_dok)
    validasi = {
        "sha256_masukan": sidik,
        "dokumen": ditemukan,
        "teks_kosong": kosong,
        "dicurigai_terpotong_batas_excel": terpotong,
        "kata_bt_total": kata_bt,
        "kata_id_total": sum(d["kata_id"] for d in per_dok),
        "kata_bt_sesuai": kata_bt == PARAM_DOK["kata_bt_diharapkan"],
        "per_dokumen": per_dok,
    }
    if kosong:
        raise ValueError(f"Ada teks Agenda yang kosong: {kosong}")

    kolom = ["doc_id", "title", "text_bt", "text_id", "genre", "subgenre", "source_link"]
    simpan_csv(agenda[kolom], P_A01_DOK)
    simpan_json(validasi, P_A01_VAL)
    t1.selesai({
        "dokumen": len(agenda),
        "kata_bt_total": validasi["kata_bt_total"],
        "kata_id_total": validasi["kata_id_total"],
        "kata_bt_sesuai": validasi["kata_bt_sesuai"],
        "dicurigai_terpotong": len(terpotong),
    })

agenda = baca_csv(P_A01_DOK)
validasi_a01 = json.loads(P_A01_VAL.read_text(encoding="utf-8"))
if not validasi_a01["kata_bt_sesuai"]:
    print(f"PERHATIAN: total kata Toba {validasi_a01['kata_bt_total']} tidak sama dengan {PARAM_DOK['kata_bt_diharapkan']}.")
tampilkan(pd.DataFrame(validasi_a01["per_dokumen"]), 10)

---
## Tahap a02. Struktur baris

**Tujuan:** mengenali struktur liturgi pada setiap baris, supaya setiap kalimat nanti tahu letaknya (bagian, subbagian, butir, dan siapa yang mengucapkannya). Struktur ini akan menjadi jangkar penyelarasan.

**Yang dikerjakan:**
1. Memecah teks per baris dan membuang baris kosong. Nomor baris asli tetap dicatat di `baris_asal`.
2. Memisahkan **penanda** di awal baris: angka Romawi (`I.`), huruf (`A.`, `a)`), angka (`1.`), atau gabungannya (`A. 1.`, `XVIII.B.`). Angka Romawi yang terbaca dengan huruf l kecil (`XVIl.`) dibaca sebagai `XVII`, tetapi penanda aslinya tetap disimpan.
3. Menentukan jenis setiap baris:

| Jenis baris | Aturan | Contoh |
|---|---|---|
| `judul_besar` | Diawali `Bagian` dan paling banyak 4 kata | `Bagian Parjolo`, `Bagian Pertama` |
| `judul_bagian` | Penanda pertamanya angka Romawi, isinya paling banyak 12 kata, dan **tidak** termasuk deret Romawi berurutan | `I. Parturena di parjamitaan jumpa ari Minggu.` |
| `butir` | Baris berpenanda lainnya, termasuk deret Romawi berurutan minimal 3 baris (misalnya Sepuluh Hukum I sampai X) | `VI. Unang ho mamunu !`, `1. Dipangido hamu ...` |
| `subjudul` | Penanda satu huruf kapital, paling banyak 10 kata, tanpa titik dua, tanda seru, tanda tanya, kurung, atau titik isian di akhir | `A. DI GAREJA` |
| `label` | Diawali label peran dari daftar di Konfigurasi, diikuti titik dua. Isi sesudah titik dua dipisah menjadi `sisa` | `Marende :`, `PELAYAN LITURGI :`, `Alusna: Na olo do hami.` |
| `isi` | Selain yang di atas | |

4. **Menggabungkan baris yang terpotong di tengah kalimat:** baris isi yang berakhir dengan koma atau titik koma, atau berakhir dengan huruf dan disusul baris berhuruf kecil, digabung dengan baris isi berikutnya. Kasus ini hampir semuanya ada di sisi Indonesia (misalnya Doa Bapa Kami dan berkat yang ditulis per baris).
5. Memberi keterangan struktur: `bagian_ke` (urutan judul dalam dokumen, dimulai dari 0 untuk teks sebelum judul pertama), `bagian_romawi`, `judul_bagian`, `subbagian`, `butir`, dan `peran` (label peran terakhir yang masih berlaku dalam bagian itu). Label nyanyian, warta, dan khotbah (`label_tidak_menurun` di Konfigurasi, misalnya `Marende :` dan `NYANYIAN BERSAMA : No...`) hanya berlaku pada barisnya sendiri, karena syair atau isinya memang tidak dicetak.

**Masukan:** `a01_agenda_dokumen.csv`.

**Keluaran yang diharapkan:**
- `a02_baris.csv`: satu baris per baris teks (sekitar 870 baris Toba dan 1.200 baris Indonesia).
- `a02_ringkasan_struktur.csv`: jumlah setiap jenis baris dan jumlah bagian per dokumen dan bahasa.

**Dipakai untuk:** tahap a03.

**Cara memeriksa:** jumlah `judul_besar` ditambah `judul_bagian` harus sama di sisi Toba dan Indonesia untuk setiap dokumen (`bagian_sama` bernilai `True`). Selisih menandakan aturan judul perlu disesuaikan, atau memang ada bagian yang hilang di salah satu sisi.

In [ ]:
P_A02_BARIS = DIR_OUT / "a02_baris.csv"
P_A02_RING = DIR_OUT / "a02_ringkasan_struktur.csv"
PARAM_ST = KONFIGURASI["struktur"]

_SATU = r"(?:[IVX][IVXl]*|[A-Za-z]|\d{1,3})\s?[.)]"
POLA_PENANDA = re.compile(rf"^(?P<p>(?:{_SATU}\s*)*{_SATU})\s+(?=\S)")
POLA_SATU_PENANDA = re.compile(r"([IVX][IVXl]*|[A-Za-z]|\d{1,3})\s?[.)]")
POLA_ROMAWI = re.compile(r"^(?=[IVX])X{0,3}(?:IX|IV|V?I{0,3})$")
POLA_LABEL = re.compile(r"^(?P<lab>[^:()\n]{1,60}?)\s*:\s*(?P<sisa>.*)$")


def nilai_romawi(s):
    nilai = {"I": 1, "V": 5, "X": 10}
    total = 0
    for i, ch in enumerate(s):
        v = nilai[ch]
        total += -v if i + 1 < len(s) and nilai[s[i + 1]] > v else v
    return total


def baca_penanda(penanda):
    romawi = huruf = angka = ""
    daftar = POLA_SATU_PENANDA.findall(penanda)
    for k, x in enumerate(daftar):
        if len(x) > 1 and re.fullmatch(r"[IVXl]+", x):
            x = x.replace("l", "I")
        if k == 0 and POLA_ROMAWI.match(x):
            romawi = x
        elif x.isdigit():
            angka = x
        elif x.isalpha() and not huruf:
            huruf = x
    return romawi, huruf, angka, len(daftar)


def klasifikasi_baris(teks_dok, bahasa):
    label_sah = set(PARAM_ST["label_peran"][bahasa])
    baris = []
    for no, mentah in enumerate(teks_dok.split("\n"), start=1):
        s = mentah.strip()
        if not s:
            continue
        r = {"baris_asal": str(no), "teks_baris": s, "penanda": "", "jenis_baris": "isi", "label": "",
             "sisa": s, "romawi": "", "huruf": "", "angka": "", "jumlah_penanda": 0}
        m = POLA_PENANDA.match(s)
        if m:
            r["penanda"] = m.group("p").strip()
            r["sisa"] = s[m.end():]
            r["romawi"], r["huruf"], r["angka"], r["jumlah_penanda"] = baca_penanda(r["penanda"])
            r["jenis_baris"] = "berpenanda"
        elif s.split()[0].lower() in PARAM_ST["judul_besar_awalan"] and len(s.split()) <= PARAM_ST["judul_besar_maks_kata"]:
            r["jenis_baris"] = "judul_besar"
        else:
            ml = POLA_LABEL.match(s)
            if ml and kunci_label(ml.group("lab")) in label_sah:
                r["jenis_baris"] = "label"
                r["label"] = s[: ml.start("sisa")].rstrip()
                r["sisa"] = ml.group("sisa").strip()
        baris.append(r)

    # Deret Romawi berurutan di baris bersebelahan adalah butir daftar, bukan judul bagian.
    idx_romawi = [i for i, r in enumerate(baris) if r["romawi"] and not r["angka"]]
    deret = set()
    i = 0
    while i < len(idx_romawi):
        j = i
        while (j + 1 < len(idx_romawi) and idx_romawi[j + 1] == idx_romawi[j] + 1
               and nilai_romawi(baris[idx_romawi[j + 1]]["romawi"]) == nilai_romawi(baris[idx_romawi[j]]["romawi"]) + 1):
            j += 1
        if j - i + 1 >= PARAM_ST["deret_daftar_min"]:
            deret.update(idx_romawi[i:j + 1])
        i = j + 1
    for k, r in enumerate(baris):
        if r["jenis_baris"] != "berpenanda":
            continue
        if r["romawi"] and not r["angka"] and k not in deret and jumlah_kata(r["sisa"]) <= PARAM_ST["judul_bagian_maks_kata"]:
            r["jenis_baris"] = "judul_bagian"
        elif (r["huruf"].isupper() and r["jumlah_penanda"] == 1 and r["penanda"].endswith(".")
              and jumlah_kata(r["sisa"]) <= PARAM_ST["subjudul_maks_kata"]
              and not re.search(r"[:!?]\s*$", r["sisa"]) and "(" not in r["sisa"] and ".." not in r["sisa"]):
            r["jenis_baris"] = "subjudul"
        else:
            r["jenis_baris"] = "butir"

    # Gabungkan baris isi yang terpotong di tengah kalimat.
    gb = PARAM_ST["gabung_baris"]
    hasil = []
    for r in baris:
        r["digabung"] = False
        if hasil:
            seb = hasil[-1]
            boleh = seb["jenis_baris"] in ("isi", "butir") and r["jenis_baris"] == "isi"
            koma = gb["akhir_koma"] and re.search(r"[,;]$", seb["teks_baris"])
            kecil = gb["lanjut_huruf_kecil"] and re.search(r"\w$", seb["teks_baris"]) and r["teks_baris"][0].islower()
            if boleh and (koma or kecil):
                seb["teks_baris"] += " " + r["teks_baris"]
                seb["sisa"] += " " + r["sisa"]
                seb["baris_asal"] = seb["baris_asal"].split("-")[0] + "-" + r["baris_asal"]
                seb["digabung"] = True
                continue
        hasil.append(r)

    # Keterangan struktur.
    tidak_menurun = set(PARAM_ST["label_tidak_menurun"][bahasa])
    bagian_ke, romawi, judul, sub, peran = 0, "", "", "", ""
    for r in hasil:
        if r["jenis_baris"] in ("judul_besar", "judul_bagian"):
            bagian_ke += 1
            romawi, judul, sub, peran = r["romawi"], r["teks_baris"], "", ""
        if r["huruf"].isupper():
            sub = r["huruf"]
        peran_baris = peran
        if r["jenis_baris"] == "label":
            peran_baris = kunci_label(r["label"].rstrip(":").strip())
            peran = "" if peran_baris in tidak_menurun else peran_baris
        huruf_kecil = r["huruf"] if r["huruf"].islower() else ""
        butir = ".".join(x for x in (huruf_kecil, r["angka"]) if x)
        if not butir and r["jenis_baris"] == "butir":
            butir = r["romawi"] or r["huruf"]
        r.update({"bagian_ke": bagian_ke, "bagian_romawi": romawi, "judul_bagian": judul,
                  "subbagian": sub, "butir": butir, "peran": peran_baris})
    return hasil


t2 = Tahap(
    "a02", "Struktur baris",
    masukan=[P_A01_DOK],
    keluaran=[P_A02_BARIS, P_A02_RING],
    parameter={"struktur": PARAM_ST, "ocr": KONFIGURASI["ocr"], "fungsi_teks": SIDIK_FUNGSI_TEKS},
)

if not t2.bisa_dilewati():
    semua = []
    for _, d in agenda.iterrows():
        for bahasa, kol in (("bbc", "text_bt"), ("ind", "text_id")):
            for urut, r in enumerate(klasifikasi_baris(d[kol], bahasa)):
                semua.append({"doc_id": d["doc_id"], "bahasa": bahasa, "urut": urut, **r})
    kolom = ["doc_id", "bahasa", "urut", "baris_asal", "jenis_baris", "penanda", "romawi", "huruf", "angka",
             "label", "sisa", "teks_baris", "digabung", "bagian_ke", "bagian_romawi", "judul_bagian",
             "subbagian", "butir", "peran"]
    baris_df = pd.DataFrame(semua)[kolom]
    ring = baris_df.groupby(["doc_id", "bahasa", "jenis_baris"]).size().unstack(fill_value=0).reset_index()
    ring["bagian"] = ring.get("judul_besar", 0) + ring.get("judul_bagian", 0)
    ring["baris_digabung"] = baris_df.groupby(["doc_id", "bahasa"])["digabung"].sum().values
    bagian = ring.pivot(index="doc_id", columns="bahasa", values="bagian")
    simpan_csv(baris_df, P_A02_BARIS)
    simpan_csv(ring, P_A02_RING)
    t2.selesai({
        "baris_bt": int((baris_df["bahasa"] == "bbc").sum()),
        "baris_id": int((baris_df["bahasa"] == "ind").sum()),
        "bagian_bt": int(bagian["bbc"].sum()),
        "bagian_id": int(bagian["ind"].sum()),
        "dokumen_bagian_tidak_sama": [d for d in bagian.index if bagian.at[d, "bbc"] != bagian.at[d, "ind"]],
        "label_bt": int(((baris_df["bahasa"] == "bbc") & (baris_df["jenis_baris"] == "label")).sum()),
        "label_id": int(((baris_df["bahasa"] == "ind") & (baris_df["jenis_baris"] == "label")).sum()),
    })

baris_a02 = baca_csv(P_A02_BARIS, kolom_angka=["urut", "bagian_ke"])
ring_a02 = baca_csv(P_A02_RING)
tabel_bagian = ring_a02.pivot(index="doc_id", columns="bahasa", values="bagian")
tabel_bagian["bagian_sama"] = tabel_bagian["bbc"] == tabel_bagian["ind"]
print("Jumlah judul (judul_besar + judul_bagian) per dokumen:")
tampilkan(tabel_bagian, 10)
print("\nContoh judul bagian dari kedua sisi:")
tampilkan(baris_a02[baris_a02["jenis_baris"] == "judul_bagian"][["doc_id", "bahasa", "bagian_ke", "teks_baris"]], 6)

---
## Tahap a03. Segmentasi kalimat

**Tujuan:** memecah isi setiap baris menjadi unit kalimat, dengan aturan yang sesuai untuk teks liturgi.

**Yang dikerjakan:**
1. Judul besar, judul bagian, dan subjudul menjadi satu unit utuh. Label peran menjadi unit sendiri, dan isi sesudah titik duanya diproses seperti baris isi.
2. Isi baris dipecah pada `.`, `!`, atau `?` yang diikuti spasi dan huruf kapital, tanda kutip pembuka, atau kurung buka. Titik setelah singkatan di Konfigurasi (misalnya `No.`, `Hal.`, `Jes.`, `Mat.`) tidak dianggap akhir kalimat.
3. **Kurung dilindungi:** tidak ada pemecahan di dalam kurung, sehingga instruksi seperti `( Baca salah satu ayat ... Lihat pada bagian A ... )` tetap utuh. Kurung yang berdiri sesudah akhir kalimat dipisah menjadi unit sendiri. Pengecualiannya, isi kurung yang lebih dari 25 kata (misalnya doa pilihan yang seluruhnya ditulis dalam kurung) tetap dipecah per kalimat.
4. **Seruan dan vokatif digabung ke kalimat sesudahnya:** unit yang diakhiri `!` dan pendek (paling banyak 3 kata), atau diawali sapaan (`Ale`, `O` di Toba; `Ya`, `Hai` di Indonesia) dan paling banyak 12 kata, digabung dengan kalimat sesudahnya dalam baris yang sama. Alasannya, Toba menulis `Ale Tuhan Debata ! Sai pargogoi ma hami ...`, sedangkan Indonesia sering menulis `Ya Tuhan, Allah, kuatkanlah kami ...` sebagai satu kalimat. Tanpa aturan ini sapaan Toba menjadi unit lepas yang tidak punya padanan. Formula seperti `Amen !` tidak digabung, kecuali dengan formula yang sama (`Haleluya ! Haleluya ! Haleluya !` menjadi satu unit).
5. Memberi tanda `seruan_digabung`, `baris_digabung`, `kurung_tidak_seimbang` (baris aslinya punya jumlah kurung buka dan tutup yang berbeda), dan `kurung_panjang_dipecah`.

**Masukan:** `a02_baris.csv`.

**Keluaran yang diharapkan:**
- `a03_unit_mentah.csv`: satu baris per unit dengan `teks_asli` (belum dinormalisasi), penanda, keterangan struktur, `jenis_awal`, dan `tanda`.

**Dipakai untuk:** tahap a04.

**Cara memeriksa:** lihat contoh unit bertanda `seruan_digabung` di bawah sel. Bila penggabungan terasa berlebihan, matikan aturan ini di Konfigurasi dan bandingkan jumlah unitnya.

**Perbedaan dengan tahap s04 notebook utama:** s04 memecah semua tanda seru dan memecah di dalam kurung. Di Agenda, kedua hal itu menghasilkan banyak sapaan dan potongan instruksi yang berdiri sendiri. Pembandingan jumlah unitnya ada di tahap a05.

In [ ]:
P_A03_UNIT = DIR_OUT / "a03_unit_mentah.csv"
PARAM_KAL = KONFIGURASI["kalimat"]
PENUTUP = "\"”’'"
PEMBUKA = "\"“‘'("


def akhir_singkatan(teks_sebelum, singkatan):
    kata = re.findall(r"\w+", teks_sebelum.lower())
    return bool(kata) and kata[-1] in singkatan


def pecah_kalimat(s, bahasa):
    singkatan = set(PARAM_KAL["singkatan"])
    n = len(s)
    # Kurung tingkat atas yang panjang dianggap transparan: kalimat di dalamnya tetap dipecah.
    transparan, tumpuk = set(), []
    for idx, ch in enumerate(s):
        if ch == "(":
            tumpuk.append(idx)
        elif ch == ")" and tumpuk:
            buka = tumpuk.pop()
            if not tumpuk and jumlah_kata(s[buka:idx]) > PARAM_KAL["kurung_panjang_min_kata"]:
                transparan.update((buka, idx))
    if len(tumpuk) == 1 and jumlah_kata(s[tumpuk[0]:]) > PARAM_KAL["kurung_panjang_min_kata"]:
        transparan.add(tumpuk[0])

    potong, kedalaman, i = [], 0, 0
    while i < n:
        ch = s[i]
        if i in transparan:
            pass
        elif ch == "(":
            kedalaman += 1
        elif ch == ")":
            kedalaman = max(0, kedalaman - 1)
            if kedalaman == 0:
                j = i + 1
                while j < n and s[j] in ".,;:!?":
                    j += 1
                k = j
                while k < n and s[k].isspace():
                    k += 1
                if k > j and k < n and (s[k].isupper() or s[k] in PEMBUKA):
                    potong.append(j)
                    i = j
                    continue
        elif ch in ".!?" and kedalaman == 0:
            j = i + 1
            while j < n and s[j] in PENUTUP + ".!?":
                j += 1
            k = j
            while k < n and s[k].isspace():
                k += 1
            if k > j and k < n and (s[k].isupper() or s[k] in PEMBUKA):
                if not (ch == "." and akhir_singkatan(s[:i], singkatan)):
                    potong.append(j)
            i = j
            continue
        i += 1

    seg, awal = [], 0
    for p in potong + [n]:
        if s[awal:p].strip():
            seg.append(s[awal:p].strip())
        awal = p

    gs = PARAM_KAL["gabung_seruan"]
    formula = set(KONFIGURASI["jenis_unit"]["formula"])
    if not gs["aktif"]:
        return seg, [False] * len(seg)
    hasil, digabung = [], []
    for x in seg:
        if hasil:
            seb = hasil[-1]
            kata_seb = re.findall(r"\w+", seb.lower())
            kata_ini = re.findall(r"\w+", x.lower())
            seruan = re.search(r"!\s*[\"”’']*$", seb) is not None
            adalah_formula = bool(kata_seb) and set(kata_seb) <= formula
            sama_formula = adalah_formula and bool(kata_ini) and set(kata_ini) <= set(kata_seb)
            pendek = len(kata_seb) <= gs["maks_kata_pendek"]
            vokatif = (bool(kata_seb) and kata_seb[0] in gs["awalan_vokatif"][bahasa]
                       and len(kata_seb) <= gs["maks_kata_vokatif"])
            if seruan and not seb.startswith("(") and not x.startswith("(") and (
                    sama_formula or (not adalah_formula and (pendek or vokatif))):
                hasil[-1] = seb + " " + x
                digabung[-1] = True
                continue
        hasil.append(x)
        digabung.append(False)
    return hasil, digabung


t3 = Tahap(
    "a03", "Segmentasi kalimat",
    masukan=[P_A02_BARIS],
    keluaran=[P_A03_UNIT],
    parameter={"kalimat": PARAM_KAL, "formula": KONFIGURASI["jenis_unit"]["formula"], "fungsi_teks": SIDIK_FUNGSI_TEKS},
)

if not t3.bisa_dilewati():
    baris_df = baca_csv(P_A02_BARIS, kolom_angka=["urut", "bagian_ke"]).sort_values(["doc_id", "bahasa", "urut"])
    unit = []
    hitung = {}

    def tambah(r, teks_asli, jenis_awal, penanda="", tanda=()):
        kunci = (r["doc_id"], r["bahasa"])
        idx = hitung.get(kunci, 0)
        hitung[kunci] = idx + 1
        tanda = list(tanda)
        if r["digabung"] == "True":
            tanda.append("baris_digabung")
        if r["sisa"].count("(") != r["sisa"].count(")"):
            tanda.append("kurung_tidak_seimbang")
        elif teks_asli.count("(") != teks_asli.count(")"):
            tanda.append("kurung_panjang_dipecah")
        unit.append({
            "unit_id": f"{r['doc_id']}_{r['bahasa']}_{idx:04d}", "doc_id": r["doc_id"], "bahasa": r["bahasa"],
            "idx": idx, "baris_asal": r["baris_asal"], "bagian_ke": r["bagian_ke"],
            "bagian_romawi": r["bagian_romawi"], "judul_bagian": r["judul_bagian"], "subbagian": r["subbagian"],
            "butir": r["butir"] if penanda else "", "penanda": penanda,
            "peran": "" if jenis_awal in ("judul_besar", "judul_bagian", "subjudul", "label_peran") else r["peran"],
            "jenis_awal": jenis_awal, "teks_asli": teks_asli, "tanda": ";".join(dict.fromkeys(tanda)),
        })

    for _, r in baris_df.iterrows():
        jb = r["jenis_baris"]
        if jb in ("judul_besar", "judul_bagian", "subjudul"):
            tambah(r, r["sisa"], jb, penanda=r["penanda"])
            continue
        if jb == "label":
            tambah(r, r["label"], "label_peran")
            if not r["sisa"]:
                continue
        segmen, digabung = pecah_kalimat(r["sisa"], r["bahasa"])
        for k, (x, g) in enumerate(zip(segmen, digabung)):
            tambah(r, x, "kalimat", penanda=r["penanda"] if k == 0 else "", tanda=["seruan_digabung"] if g else [])

    unit_df = pd.DataFrame(unit)
    simpan_csv(unit_df, P_A03_UNIT)
    tanda_hitung = Counter(t for x in unit_df["tanda"] for t in x.split(";") if t)
    t3.selesai({
        "unit_bt": int((unit_df["bahasa"] == "bbc").sum()),
        "unit_id": int((unit_df["bahasa"] == "ind").sum()),
        "tanda": dict(tanda_hitung.most_common()),
    })

unit_a03 = baca_csv(P_A03_UNIT, kolom_angka=["idx", "bagian_ke"])
print(unit_a03.groupby(["doc_id", "bahasa"]).size().unstack())
print("\nContoh unit bertanda seruan_digabung:")
contoh_seruan = unit_a03[unit_a03["tanda"].str.contains("seruan_digabung")]
tampilkan(contoh_seruan.sample(min(6, len(contoh_seruan)), random_state=KONFIGURASI["seed"])[["unit_id", "teks_asli"]], 6)

---
## Tahap a04. Normalisasi OCR terbatas dan jenis unit

**Tujuan:** menghasilkan teks unit yang siap dipakai untuk penyelarasan, tanpa kehilangan teks aslinya, dan memberi jenis pada setiap unit.

**Yang dikerjakan:**
1. Menerapkan `normalkan_ocr` (lihat tabel aturan di Bagian A) pada setiap unit. Hasilnya di kolom `teks`, sedangkan `teks_asli` tidak diubah. Nama aturan yang mengubah unit dicatat di `koreksi_ocr`.
2. Menentukan `jenis_unit`:

| Jenis unit | Arti | Contoh |
|---|---|---|
| `judul_besar`, `judul_bagian`, `subjudul` | Dari tahap a02 | `TATA KEBAKTIAN MINGGU` |
| `label_peran` | Rubrik peran pengucap | `Marende :`, `PELAYAN LITURGI :` |
| `rujukan_ayat` | Kurung yang hanya berisi rujukan ayat: nama kitab (boleh berawalan 1 sampai 3 atau I sampai III) yang langsung diikuti angka, dan sisanya hanya angka, pemisah, dan huruf penanda bagian ayat | `( Jes. 62, 11 ).`, `(I Petrus 1: 3a).`, `(Yesaya 45: 8. Mazmur 19: 2).` |
| `instruksi` | Kurung lain yang berdiri sendiri | `( Hanya pelayan yang mendoakan )` |
| `isian` | Tidak berisi kata selain `No`, misalnya titik isian | `No...` |
| `formula` | Hanya berisi kata formula liturgi | `Amen.`, `Amin - Amin - Amin .`, `Haleluya ! Haleluya !` |
| `isi` | Kalimat liturgi biasa | |

3. Menambah tanda: `singkatan_lanjutan` (unit menyingkat teks baku dengan `d.u.` atau `dst.`, misalnya `Ahu porsea d.u.`), `titik_isian`, dan `memuat_rujukan` (rujukan ayat di dalam kalimat). Rujukan ayat juga disalin ke kolom `rujukan_ayat`.
4. Menghitung `jumlah_kata` dari teks hasil normalisasi.

**Masukan:** `a03_unit_mentah.csv`.

**Keluaran yang diharapkan:**
- `a04_unit_agenda.csv`: **keluaran utama** notebook ini, satu baris per unit. Kolomnya: `unit_id`, `doc_id`, `bahasa`, `idx`, `baris_asal`, `bagian_ke`, `bagian_romawi`, `judul_bagian`, `subbagian`, `butir`, `penanda`, `peran`, `jenis_unit`, `teks_asli`, `teks`, `koreksi_ocr`, `rujukan_ayat`, `tanda`, `jumlah_kata`.
- `a04_log_ocr.csv`: setiap unit yang berubah, dengan teks sebelum dan sesudah.

**Dipakai untuk:** tahap a05, dan nanti sebagai masukan penyelarasan.

**Cara memeriksa:** baca beberapa baris `a04_log_ocr.csv`. Koreksi harus terlihat jelas benar. Bila ada koreksi yang salah, hapus kata itu dari `ganti_kata` di Konfigurasi.

**Catatan:** unit `isi` yang berakhir dengan `d.u.` atau `dst.` tidak memuat teks lengkapnya. Saat penyelarasan, unit bertanda `singkatan_lanjutan` sebaiknya hanya dipasangkan dengan unit yang juga menyingkat, atau dikeluarkan.

In [ ]:
P_A04_UNIT = DIR_OUT / "a04_unit_agenda.csv"
P_A04_LOG = DIR_OUT / "a04_log_ocr.csv"
PARAM_JU = KONFIGURASI["jenis_unit"]

POLA_NAMA_KITAB = re.compile(r"(?<![A-Za-z])(?:(?:[1-3]|I{1,3})\s?)?[A-Z][A-Za-z]{1,20}(?:-[a-z]+)?\.?")
POLA_KATA_SAMBUNG_RUJUKAN = re.compile(r"(?<![A-Za-z])(?:dari|para|dan|bnd)(?![A-Za-z])")
POLA_KURUNG = re.compile(r"\(([^()]*)\)")


def adalah_rujukan(isi):
    # Isi kurung dianggap rujukan ayat bila diawali nama kitab dan memuat angka, dan setelah nama kitab serta
    # kata sambung (dari, para, dan) dibuang, sisanya hanya angka, pemisah, dan huruf tunggal penanda bagian
    # ayat (3a, 4b, 6D). Angka 1 yang terbaca huruf l (la, lb, lla) dibaca sebagai angka, hanya untuk pengenalan.
    isi = isi.strip()
    isi = re.sub(r"(?<![A-Za-z])l(l?)([ab]?)(?![A-Za-z])", lambda m: "1" + ("1" if m.group(1) else "") + m.group(2), isi)
    if len(isi) > 160 or not re.match(r"(?:dari\s+)?(?:(?:[1-3]|I{1,3})\s?)?[A-Z][a-z]", isi) or not re.search(r"\d", isi):
        return False
    sisa = POLA_KATA_SAMBUNG_RUJUKAN.sub(" ", POLA_NAMA_KITAB.sub(" ", isi))
    return re.fullmatch(r"(?:[\d\s,.;:\u2013\-]|(?<![A-Za-z])[A-Za-z](?![A-Za-z]))*", sisa) is not None


def rujukan_dalam(teks):
    return [re.sub(r"\s+", " ", m.group(1)).strip() for m in POLA_KURUNG.finditer(teks) if adalah_rujukan(m.group(1))]


def tentukan_jenis(teks, jenis_awal):
    if jenis_awal != "kalimat":
        return jenis_awal
    t = teks.strip()
    if t.startswith("("):
        m = re.fullmatch(r"\(([^()]*)\)\s*[.,;]?", t)
        if m and adalah_rujukan(m.group(1)):
            return "rujukan_ayat"
        tertutup = re.search(r"\)\s*[.,;:!?]*$", t) is not None
        if (tertutup and t.count("(") <= t.count(")")) or (
                t.count("(") > t.count(")") and jumlah_kata(t) <= KONFIGURASI["kalimat"]["kurung_panjang_min_kata"]):
            return "instruksi"
    kata = re.findall(r"[^\W\d_]+", t.lower())
    if not [k for k in kata if k != "no"]:
        return "isian"
    if set(kata) <= set(PARAM_JU["formula"]):
        return "formula"
    return "isi"


t4 = Tahap(
    "a04", "Normalisasi OCR dan jenis unit",
    masukan=[P_A03_UNIT],
    keluaran=[P_A04_UNIT, P_A04_LOG],
    parameter={"ocr": KONFIGURASI["ocr"], "jenis_unit": PARAM_JU,
               "kurung_panjang_min_kata": KONFIGURASI["kalimat"]["kurung_panjang_min_kata"],
               "fungsi_teks": SIDIK_FUNGSI_TEKS},
)

if not t4.bisa_dilewati():
    u = baca_csv(P_A03_UNIT, kolom_angka=["idx", "bagian_ke"])
    teks_baru, koreksi, jenis, rujukan, tanda_baru = [], [], [], [], []
    for _, r in u.iterrows():
        t, cat = normalkan_ocr(r["teks_asli"], r["bahasa"])
        j = tentukan_jenis(t, r["jenis_awal"])
        tanda = [x for x in r["tanda"].split(";") if x]
        if any(re.search(p, t) for p in PARAM_JU["penanda_singkatan_lanjutan"]):
            tanda.append("singkatan_lanjutan")
        if "..." in t:
            tanda.append("titik_isian")
        rj = rujukan_dalam(t)
        if rj and j != "rujukan_ayat":
            tanda.append("memuat_rujukan")
        teks_baru.append(t)
        koreksi.append(";".join(cat))
        jenis.append(j)
        rujukan.append(" | ".join(rj))
        tanda_baru.append(";".join(dict.fromkeys(tanda)))
    u["teks"] = teks_baru
    u["koreksi_ocr"] = koreksi
    u["jenis_unit"] = jenis
    u["rujukan_ayat"] = rujukan
    u["tanda"] = tanda_baru
    u["jumlah_kata"] = u["teks"].map(jumlah_kata)
    kolom = ["unit_id", "doc_id", "bahasa", "idx", "baris_asal", "bagian_ke", "bagian_romawi", "judul_bagian",
             "subbagian", "butir", "penanda", "peran", "jenis_unit", "teks_asli", "teks", "koreksi_ocr",
             "rujukan_ayat", "tanda", "jumlah_kata"]
    u = u[kolom]
    log = u.loc[u["koreksi_ocr"] != "", ["unit_id", "bahasa", "koreksi_ocr", "teks_asli", "teks"]]
    simpan_csv(u, P_A04_UNIT)
    simpan_csv(log, P_A04_LOG)
    aturan = Counter(x.split(":")[0] for c in log["koreksi_ocr"] for x in c.split(";"))
    t4.selesai({
        "unit": len(u),
        "unit_dikoreksi_ocr": {"bbc": int((log["bahasa"] == "bbc").sum()), "ind": int((log["bahasa"] == "ind").sum())},
        "aturan_ocr": dict(aturan.most_common()),
        "jenis_unit": u.groupby(["jenis_unit", "bahasa"]).size().unstack(fill_value=0).to_dict("index"),
    })

unit_agenda = baca_csv(P_A04_UNIT, kolom_angka=["idx", "bagian_ke", "jumlah_kata"])
log_ocr = baca_csv(P_A04_LOG)
print("Jenis unit per bahasa:")
tampilkan(unit_agenda.groupby(["jenis_unit", "bahasa"]).size().unstack(fill_value=0), 12)
print("\nContoh koreksi OCR:")
tampilkan(log_ocr.sample(min(8, len(log_ocr)), random_state=KONFIGURASI["seed"]), 8)

---
## Tahap a05. Pemeriksaan dan ringkasan

**Tujuan:** membuktikan bahwa segmentasi tidak menghilangkan atau mengubah teks, mengukur kesejajaran struktur kedua sisi, dan menyiapkan sampel untuk dibaca manusia.

**Yang dikerjakan:**
1. **Keutuhan teks.** Untuk setiap dokumen dan bahasa, gabungan `penanda` dan `teks_asli` semua unit (tanpa spasi) harus sama persis dengan teks dokumen (tanpa spasi). Jumlah kata juga harus sama.
2. **Kesejajaran bagian.** Tabel per `bagian_ke` yang memuat judul, jumlah unit isi, jumlah label, dan jumlah kata di kedua sisi. Nomor Romawi yang berbeda (salah ketik di sumber) dicatat, tetapi pencocokan tetap memakai urutan `bagian_ke`.
3. **Pembanding segmentasi lama.** Jumlah kalimat menurut fungsi `pecah_kalimat` tahap s04 notebook utama (yang juga dipakai uji penyelarasan v2), dibandingkan dengan jumlah unit notebook ini.
4. **Duplikat (informasi saja).** Jumlah unit `isi` dan `formula` yang teksnya muncul lebih dari sekali, setelah huruf kecil dan spasi diseragamkan. Tidak ada unit yang dibuang di sini. Deduplikasi dikerjakan setelah penyelarasan.
5. **Sampel pemeriksaan.** Sampel acak bertingkat per jenis unit dan bahasa, ditambah semua unit bertanda `kurung_tidak_seimbang` dan unit terpanjang. Kolom `catatan_pemeriksa` dibiarkan kosong untuk diisi.

**Masukan:** `a01_agenda_dokumen.csv`, `a04_unit_agenda.csv`.

**Keluaran yang diharapkan:**
- `a05_ringkasan.json`: hasil semua pemeriksaan.
- `a05_per_bagian.csv`: satu baris per bagian per dokumen, kedua bahasa berdampingan.
- `a05_sampel_periksa.csv`: sampel unit untuk dibaca.

**Dipakai untuk:** laporan teknis, dan dasar keputusan sebelum penyelarasan.

**Cara memeriksa:** `keutuhan_teks` harus `utuh` untuk ke-12 kombinasi dokumen dan bahasa. Buka `a05_sampel_periksa.csv` dan baca unitnya. Bila ada pola pemecahan yang salah, catat contohnya untuk memperbaiki aturan.

**Catatan angka:** `kata_per_bahasa` dihitung dari kolom `teks` (hasil normalisasi, tanpa penanda), sehingga lebih kecil dari 35.097 kata Toba di tahap a01. Penanda butir seperti `I.` atau `A. 1.` tidak dihitung, dan huruf berspasi seperti `A m e n` dihitung sebagai satu kata. Pemeriksaan keutuhan tetap memakai `teks_asli` dan `penanda`, sehingga jumlah katanya sama persis dengan dokumen.

In [ ]:
P_A05_JSON = DIR_OUT / "a05_ringkasan.json"
P_A05_BAGIAN = DIR_OUT / "a05_per_bagian.csv"
P_A05_SAMPEL = DIR_OUT / "a05_sampel_periksa.csv"
PARAM_PR = KONFIGURASI["pemeriksaan"]

SINGKATAN_S04 = set(PARAM_PR["singkatan_s04"])
POLA_AKHIR_S04 = re.compile(r'(?<=[.!?])[\s\n]+(?=[A-ZÀ-Ý"\'“‘])')
POLA_DESIMAL_S04 = re.compile(r"\d+\.\d+")


def pecah_kalimat_s04(teks):
    """Salinan fungsi tahap s04 notebook utama, hanya untuk pembanding."""
    simpanan = {}

    def lindungi(m):
        kunci = f"__DES{len(simpanan)}__"
        simpanan[kunci] = m.group(0)
        return kunci

    terlindung = POLA_DESIMAL_S04.sub(lindungi, teks)
    kalimat, penampung = [], ""
    for potong in POLA_AKHIR_S04.split(terlindung):
        potong = potong.strip()
        if not potong:
            continue
        if penampung:
            kata = re.findall(r"\b\w+\b", penampung.rstrip(".!?").rstrip().lower())
            if kata and kata[-1] in SINGKATAN_S04:
                penampung = penampung + " " + potong
                continue
            kalimat.append(penampung)
        penampung = potong
    if penampung:
        kalimat.append(penampung)
    hasil = []
    for k in kalimat:
        hasil.extend(b.strip() for b in re.split(r"\n+", k) if b.strip())
    return hasil


def kunci_duplikat(teks):
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s]", " ", teks.lower())).strip()


t5 = Tahap(
    "a05", "Pemeriksaan dan ringkasan",
    masukan=[P_A01_DOK, P_A04_UNIT],
    keluaran=[P_A05_JSON, P_A05_BAGIAN, P_A05_SAMPEL],
    parameter={"pemeriksaan": PARAM_PR, "seed": KONFIGURASI["seed"], "fungsi_teks": SIDIK_FUNGSI_TEKS},
)

if not t5.bisa_dilewati():
    dok = baca_csv(P_A01_DOK)
    u = baca_csv(P_A04_UNIT, kolom_angka=["idx", "bagian_ke", "jumlah_kata"]).sort_values(["doc_id", "bahasa", "idx"])

    # 1. Keutuhan teks
    keutuhan, pembanding = {}, []
    for _, d in dok.iterrows():
        for bahasa, kol in (("bbc", "text_bt"), ("ind", "text_id")):
            g = u[(u["doc_id"] == d["doc_id"]) & (u["bahasa"] == bahasa)]
            gabungan = " ".join(f"{p} {t}" for p, t in zip(g["penanda"], g["teks_asli"]))
            sama = re.sub(r"\s", "", gabungan) == re.sub(r"\s", "", d[kol])
            kata_sama = jumlah_kata(gabungan) == jumlah_kata(d[kol])
            keutuhan[f"{d['doc_id']}:{bahasa}"] = "utuh" if sama and kata_sama else "BERBEDA"
            isi = g[~g["jenis_unit"].isin(["label_peran", "judul_besar", "judul_bagian", "subjudul"])]
            pembanding.append({"doc_id": d["doc_id"], "bahasa": bahasa,
                               "kalimat_s04": len(pecah_kalimat_s04(d[kol])),
                               "unit_semua": len(g), "unit_isi": int((g["jenis_unit"] == "isi").sum()),
                               "unit_tanpa_judul_label": len(isi)})

    # 2. Kesejajaran bagian
    agregat = (u.assign(isi=(u["jenis_unit"] == "isi").astype(int), label=(u["jenis_unit"] == "label_peran").astype(int))
                 .groupby(["doc_id", "bahasa", "bagian_ke"])
                 .agg(romawi=("bagian_romawi", "first"), judul=("judul_bagian", "first"), unit=("unit_id", "size"),
                      unit_isi=("isi", "sum"), label=("label", "sum"), kata=("jumlah_kata", "sum"))
                 .reset_index())
    bt = agregat[agregat["bahasa"] == "bbc"].drop(columns="bahasa").set_index(["doc_id", "bagian_ke"])
    idn = agregat[agregat["bahasa"] == "ind"].drop(columns="bahasa").set_index(["doc_id", "bagian_ke"])
    per_bagian = bt.join(idn, how="outer", lsuffix="_bt", rsuffix="_id").reset_index()
    per_bagian["rasio_kata_bt_id"] = (per_bagian["kata_bt"] / per_bagian["kata_id"].clip(lower=1)).round(3)
    per_bagian["romawi_berbeda"] = per_bagian["romawi_bt"].fillna("") != per_bagian["romawi_id"].fillna("")
    hanya_satu_sisi = per_bagian[per_bagian["judul_bt"].isna() | per_bagian["judul_id"].isna()]

    # 4. Duplikat (informasi)
    dup = {}
    for bahasa in ("bbc", "ind"):
        g = u[(u["bahasa"] == bahasa) & u["jenis_unit"].isin(["isi", "formula"])]
        k = g["teks"].map(kunci_duplikat)
        frek = k.value_counts()
        dup[bahasa] = {"unit": len(g), "teks_unik": int(frek.size),
                       "unit_dalam_kelompok_duplikat": int(frek[frek > 1].sum()),
                       "terbanyak": frek.head(10).to_dict()}

    # 5. Sampel pemeriksaan
    bagian_sampel = []
    for (_, _), g in u.groupby(["jenis_unit", "bahasa"]):
        bagian_sampel.append(g.sample(min(PARAM_PR["sampel_per_jenis"], len(g)), random_state=KONFIGURASI["seed"])
                             .assign(alasan_sampel="acak_per_jenis"))
    bagian_sampel.append(u[u["tanda"].str.contains("kurung_tidak_seimbang")].assign(alasan_sampel="kurung_tidak_seimbang"))
    for bahasa in ("bbc", "ind"):
        bagian_sampel.append(u[u["bahasa"] == bahasa].nlargest(PARAM_PR["unit_terpanjang"], "jumlah_kata")
                             .assign(alasan_sampel="terpanjang"))
    sampel = (pd.concat(bagian_sampel).drop_duplicates("unit_id")
              [["alasan_sampel", "unit_id", "doc_id", "bahasa", "bagian_ke", "judul_bagian", "jenis_unit",
                "peran", "penanda", "teks_asli", "teks", "tanda"]]
              .sort_values(["alasan_sampel", "unit_id"]))
    sampel["catatan_pemeriksa"] = ""

    tanda = {b: dict(Counter(t for x in u.loc[u["bahasa"] == b, "tanda"] for t in x.split(";") if t).most_common())
             for b in ("bbc", "ind")}
    ringkasan = {
        "keutuhan_teks": keutuhan,
        "semua_utuh": all(v == "utuh" for v in keutuhan.values()),
        "unit_per_bahasa": u["bahasa"].value_counts().to_dict(),
        "jenis_unit": u.groupby(["bahasa", "jenis_unit"]).size().unstack(fill_value=0).to_dict("index"),
        "kata_per_bahasa": u.groupby("bahasa")["jumlah_kata"].sum().to_dict(),
        "panjang_unit_isi": u[u["jenis_unit"] == "isi"].groupby("bahasa")["jumlah_kata"]
                             .describe().round(2).to_dict("index"),
        "bagian": {"pasangan_bagian": len(per_bagian), "hanya_satu_sisi": len(hanya_satu_sisi),
                   "romawi_berbeda": per_bagian.loc[per_bagian["romawi_berbeda"],
                                                    ["doc_id", "bagian_ke", "romawi_bt", "romawi_id"]].to_dict("records")},
        "pembanding_s04": pembanding,
        "duplikat": dup,
        "tanda": tanda,
        "sampel_periksa": int(len(sampel)),
    }
    simpan_json(ringkasan, P_A05_JSON)
    simpan_csv(per_bagian, P_A05_BAGIAN)
    simpan_csv(sampel, P_A05_SAMPEL)
    t5.selesai({
        "semua_utuh": ringkasan["semua_utuh"],
        "unit_bt": ringkasan["unit_per_bahasa"].get("bbc", 0),
        "unit_id": ringkasan["unit_per_bahasa"].get("ind", 0),
        "bagian_hanya_satu_sisi": len(hanya_satu_sisi),
        "romawi_berbeda": len(ringkasan["bagian"]["romawi_berbeda"]),
        "sampel_periksa": ringkasan["sampel_periksa"],
    })

ringkasan_a05 = json.loads(P_A05_JSON.read_text(encoding="utf-8"))
per_bagian_a05 = baca_csv(P_A05_BAGIAN)
print("Keutuhan teks:", "semua utuh" if ringkasan_a05["semua_utuh"] else ringkasan_a05["keutuhan_teks"])
print("Nomor Romawi berbeda (salah ketik di sumber):", ringkasan_a05["bagian"]["romawi_berbeda"])
print("\nPembanding dengan segmentasi s04 notebook utama:")
tampilkan(pd.DataFrame(ringkasan_a05["pembanding_s04"]), 12)
print("\nDuplikat unit isi dan formula (informasi, tidak dibuang):")
tampilkan(pd.DataFrame({b: {k: v for k, v in d.items() if k != "terbanyak"} for b, d in ringkasan_a05["duplikat"].items()}), 5)
print("\nBagian dengan rasio kata Toba terhadap Indonesia paling jauh dari 1:")
pb = per_bagian_a05.assign(r=pd.to_numeric(per_bagian_a05["rasio_kata_bt_id"]))
pb["jarak"] = (np.log(pb["r"].clip(lower=0.01))).abs()
tampilkan(pb.sort_values("jarak", ascending=False)[["doc_id", "bagian_ke", "judul_bt", "judul_id", "kata_bt", "kata_id", "rasio_kata_bt_id"]], 8)

---
# Bagian C. Ringkasan, keterbatasan, dan langkah berikutnya

Sel berikut menampilkan status setiap tahap dan angka kuncinya. Salin tabel ini ke catatan penelitian setiap kali notebook dijalankan ulang.

In [ ]:
baris = []
for kode_tahap in ["a01", "a02", "a03", "a04", "a05"]:
    meta_path = DIR_META / f"{kode_tahap}.json"
    meta = json.loads(meta_path.read_text(encoding="utf-8")) if meta_path.exists() else {}
    baris.append({"tahap": kode_tahap, "nama": meta.get("nama", ""), "status": STATUS_TAHAP.get(kode_tahap, "belum dijalankan"),
                  "waktu": meta.get("waktu", ""), "ringkasan": json.dumps(meta.get("ringkasan", {}), ensure_ascii=False)})
tampilkan(pd.DataFrame(baris), 10)

## Keterbatasan yang perlu disebut di laporan

1. **Aturan berbasis pola, belum dinilai manusia.** Segmentasi ini belum diukur terhadap segmentasi emas. Sampel di `a05_sampel_periksa.csv` perlu dibaca oleh penutur sebelum angka unit dilaporkan sebagai segmentasi yang valid.
2. **Instruksi tanpa kurung tidak dikenali.** Rubrik seperti `Jahaon sada ayat na tongon tu ganup Minggu ...` atau `Baca salah satu doa ...` yang tidak ditulis dalam kurung tetap berjenis `isi`.
3. **Daftar label peran disusun dari data ini.** Label yang tidak ada di daftar dianggap isi. Daftar perlu ditinjau bila ada sumber Agenda lain.
4. **Koreksi OCR hanya untuk sisi Indonesia** (kecuali satu koreksi Toba) dan hanya berdasarkan daftar eksplisit. Masih ada salah baca yang tidak terdaftar, dan dugaan salah ketik Toba tidak dikoreksi.
5. **Penggabungan seruan adalah keputusan rancangan.** Unit yang diawali sapaan bisa memuat dua kalimat (misalnya `Ya Tuhan, kasihanilah kami ! ...`). Aturan ini bisa dimatikan di Konfigurasi.
6. **Teks yang disingkat** (`d.u.`, `dst.`) tidak berisi teks lengkap. Unitnya ditandai `singkatan_lanjutan`, tetapi tidak dilengkapi.
7. **Hak cipta.** Keluaran memuat teks Agenda HKBP. Simpan hanya di Drive tim sampai izin redistribusinya jelas.

## Langkah berikutnya

1. Membaca `a05_sampel_periksa.csv` dan memperbaiki aturan bila perlu.
2. Penyelarasan berjangkar: pasangan dicari hanya di dalam bagian yang sama (`bagian_ke`), lalu di dalam butir bernomor yang sama, dengan pemrograman dinamis LaBSE dari notebook arsip v2.
3. Deduplikasi, pemeriksaan kebocoran, dan pembagian train/dev/test per bagian.
4. Pemeriksaan sampel pasangan oleh manusia.